# ToolOutputMixin

`ToolOutputMixin` marks objects that tools may return directly.

When a custom tool is invoked with a `ToolCall`, outputs that do not inherit from `ToolOutputMixin` are automatically converted to strings and wrapped in a `ToolMessage`.

In [2]:
from langchain_core.messages.tool import ToolOutputMixin
from langchain_core.tools import tool


class DirectOutput(ToolOutputMixin):
    def __init__(self, value: str):
        self.value = value

    def __repr__(self) -> str:
        return f"DirectOutput(value={self.value!r})"


@tool
def normal_tool(text: str) -> str:
    """Return a normal string output."""
    return text.upper()


@tool
def direct_tool(text: str) -> DirectOutput:
    """Return an object marked as direct tool output."""
    return DirectOutput(text.upper())


normal_result = normal_tool.invoke(
    {
        "name": "normal_tool",
        "args": {"text": "hello"},
        "id": "call-1",
        "type": "tool_call"
    }
)

direct_result = direct_tool.invoke(
    {
        "name": "direct_tool",
        "args": {"text": "hello"},
        "id": "call-2",
        "type": "tool_call"
    }
)


print(normal_result) # Automatically wrapped in ToolMessage
print(type(normal_result)) # ToolMessage

print(direct_result) # Returned directly
print(type(direct_result)) # DirectOutput

print(isinstance(normal_result, ToolMessage)) # True
print(isinstance(direct_result, ToolOutputMixin)) # True

# Objects that do not inherit from ToolOutputMixin are wrapped in a ToolMessage, while objects inheriting from it are returned directly.

content='HELLO' name='normal_tool' tool_call_id='call-1'
<class 'langchain_core.messages.tool.ToolMessage'>
DirectOutput(value='HELLO')
<class '__main__.DirectOutput'>
True
True


# ToolMessage: `BaseMessage`, `ToolOutputMixin`

`ToolMessage` represents the result of a tool execution returned to a chat model.

The `tool_call_id` field connects the tool result to the corresponding tool-call request. This is especially important when a model requests multiple tools in parallel.

**Syntax**

```python
ToolMessage(
    self,
    content: str | list[str | dict[Any, Any]] | None = None, # Tool-result content sent to the model
    content_blocks: list[types.ContentBlock] | None = None, # Standardized content blocks
    **kwargs: Any # Additional ToolMessage fields
)
```

## Fields

1. `tool_call_id`:`str`:= Stores the identifier of the tool call to which the message responds.

2. `type`:`Literal["tool"]`:= Stores the message type used during serialization and deserialization. Its default value is `"tool"`.

3. `artifact`:`Any`:= Stores the complete tool output or additional data that should not be sent directly to the model. Its default value is `None`.

4. `status`:`Literal["success", "error"]`:= Stores whether the tool execution succeeded or failed. Its default value is `"success"`.

5. `additional_kwargs`:`dict[Any, Any]`:= Stores additional message data inherited from `BaseMessage`.

   This field is currently not used by `ToolMessage`.

   ```python
   additional_kwargs: dict[Any, Any] = Field(
       default_factory=dict,
       repr=False
   )
   ```

6. `response_metadata`:`dict[Any, Any]`:= Stores response metadata inherited from `BaseMessage`.

   This field is currently not used by `ToolMessage`.

   ```python
   response_metadata: dict[Any, Any] = Field(
       default_factory=dict,
       repr=False
   )
   ```

## Validators

1. `coerce_args`:= Converts message content and the tool-call identifier into supported types.

   Tuple content is converted to a list. Non-string content values and unsupported list elements are converted to strings when possible. Numeric and UUID tool-call identifiers are converted to strings.

   ```python
   @model_validator(mode="before")
   @classmethod
   coerce_args(
       cls,
       values: dict[str, Any] # ToolMessage values to validate and normalize
   ) -> dict[str, Any]
   ```

In [3]:
from langchain_core.messages import ToolMessage


tool_message = ToolMessage(
    content=(42, " items found"), # Tuple becomes a list and 42 becomes "42"
    tool_call_id=101, # Numeric ID becomes "101"
    artifact={
        "database_rows": [10, 20, 12],
        "total": 42
    }, # Complete tool output kept outside model-facing content
    status="success", # Indicates successful tool execution
    additional_kwargs={
        "tool_name": "count_items"
    }, # Additional inherited message data
    response_metadata={
        "execution_time_ms": 5
    } # Inherited response metadata
)


print("Content:", tool_message.content)
print("Content type:", type(tool_message.content))
print("First content value type:", type(tool_message.content[0]))

print("Tool call ID:", tool_message.tool_call_id)
print("Tool call ID type:", type(tool_message.tool_call_id))

print("Message type:", tool_message.type)
print("Artifact:", tool_message.artifact)
print("Status:", tool_message.status)
print("Additional kwargs:", tool_message.additional_kwargs)
print("Response metadata:", tool_message.response_metadata)

Content: ['42', ' items found']
Content type: <class 'list'>
First content value type: <class 'str'>
Tool call ID: 101
Tool call ID type: <class 'str'>
Message type: tool
Artifact: {'database_rows': [10, 20, 12], 'total': 42}
Status: success
Additional kwargs: {'tool_name': 'count_items'}
Response metadata: {'execution_time_ms': 5}


# ToolMessageChunk: `ToolMessage`, `BaseMessageChunk`

`ToolMessageChunk` represents a partial tool-result message produced during streaming.

Compatible chunks can be combined while preserving the tool-call identifier and merging their content, artifact, metadata, and execution status.

**Syntax**

```python
ToolMessageChunk(
    self,
    content: str | list[str | dict[Any, Any]] | None = None, # Raw partial tool-result content
    content_blocks: list[types.ContentBlock] | None = None, # Standardized content blocks
    **kwargs: Any # Additional ToolMessageChunk fields
)
```

## Fields

1. `type`:`Literal["ToolMessageChunk"]`:= Stores the chunk-specific message type used during serialization and deserialization. Its default value is `"ToolMessageChunk"`.

## Methods
1. `__add__`:= Combines the current tool-message chunk with another compatible message chunk.

   When two `ToolMessageChunk` objects are combined, their `tool_call_id` values must match. Otherwise, a `ValueError` is raised.

   If either chunk has an `"error"` status, the resulting chunk also has an `"error"` status.

   ```python
   __add__(
       self,
       other: Any # Message chunk or another supported value to combine
   ) -> BaseMessageChunk
   ```

In [4]:
from langchain_core.messages import ToolMessageChunk


chunk1 = ToolMessageChunk(
    content="Calculation ", # First partial tool result
    tool_call_id="call-101", # Identifier of the related tool call
    artifact="Raw output part 1. ", # First partial artifact
    status="success", # First chunk completed successfully
    additional_kwargs={
        "tool_name": "calculator" # Additional data from first chunk
    },
    response_metadata={
        "chunk_number": 1 # Metadata from first chunk
    }
)

chunk2 = ToolMessageChunk(
    content="failed.", # Second partial tool result
    tool_call_id="call-101", # Must match the first chunk
    artifact="Raw output part 2.", # Second partial artifact
    status="error", # Error status affects the combined chunk
    additional_kwargs={
        "error_code": 500 # Additional data from second chunk
    },
    response_metadata={
        "completed": True # Metadata from second chunk
    }
)


combined_chunk = chunk1 + chunk2 # Combine compatible tool-message chunks


print("Content:", combined_chunk.content) # Merged tool-result content
print("Tool call ID:", combined_chunk.tool_call_id) # Preserved tool-call identifier
print("Type:", combined_chunk.type) # Displays "ToolMessageChunk"
print("Artifact:", combined_chunk.artifact) # Merged artifact
print("Status:", combined_chunk.status) # Displays "error"
print("Additional kwargs:", combined_chunk.additional_kwargs) # Merged additional data
print("Response metadata:", combined_chunk.response_metadata) # Merged metadata


try:
    different_chunk = ToolMessageChunk(
        content="Another result",
        tool_call_id="call-202" # Different tool-call identifier
    )

    result = chunk1 + different_chunk # Raises ValueError

except ValueError as error:
    print("ValueError:", error) # Display mismatched tool-call ID error

Content: Calculation failed.
Tool call ID: call-101
Type: ToolMessageChunk
Artifact: Raw output part 1. Raw output part 2.
Status: error
Additional kwargs: {'tool_name': 'calculator', 'error_code': 500}
Response metadata: {'chunk_number': 1, 'completed': True}
ValueError: Cannot concatenate ToolMessageChunks with different names.


# ToolCall: `TypedDict`
`ToolCall` represents an AI model's request to execute a tool.
## Fields
1. `name`:`str`:= Stores the name of the tool to invoke.
2. `args`:`dict[str, Any]`:= Stores the arguments passed to the tool.
3. `id`:`str | None`:= Stores an identifier used to associate the tool call with its result.
4. `type`:`NotRequired[Literal["tool_call"]]`:= Stores the optional discriminator for the tool-call structure.

In [5]:
from langchain_core.messages import ToolCall


tool_request: ToolCall = {
    "name": "add_numbers", # Name of the tool to execute
    "args": {
        "first": 10,
        "second": 20
    }, # Arguments passed to the tool
    "id": "call-101", # ID used to connect the call with its result
    "type": "tool_call" # Optional tool-call discriminator
}


print("Tool name:", tool_request["name"])
print("Arguments:", tool_request["args"])
print("Tool-call ID:", tool_request["id"])
print("Type:", tool_request["type"])

Tool name: add_numbers
Arguments: {'first': 10, 'second': 20}
Tool-call ID: call-101
Type: tool_call


# ToolCallChunk: `TypedDict`

`ToolCallChunk` represents a partial tool call produced during streaming.

When chunks are merged, their string fields are concatenated. Chunks are merged only when their non-null `index` values match.

## Fields

1. `name`:`str | None`:= Stores a partial or complete tool name.

2. `args`:`str | None`:= Stores partial tool arguments as a JSON-compatible string.

3. `id`:`str | None`:= Stores the tool-call identifier.

4. `index`:`int | None`:= Stores the position of the tool call in a sequence and is used when merging chunks.

5. `type`:`NotRequired[Literal["tool_call_chunk"]]`:= Stores the optional discriminator for the tool-call-chunk structure.

In [6]:
from langchain_core.messages import ToolCallChunk


chunk1: ToolCallChunk = {
    "name": "add_", # Partial tool name
    "args": '{"first": 10,', # Partial JSON arguments
    "id": "call-101", # Tool-call identifier
    "index": 0, # Position of the tool call
    "type": "tool_call_chunk" # Optional discriminator
}

chunk2: ToolCallChunk = {
    "name": "numbers", # Remaining tool name
    "args": '"second": 20}', # Remaining JSON arguments
    "id": None, # ID is usually provided only in the first chunk
    "index": 0, # Same index indicates the same tool call
    "type": "tool_call_chunk"
}


merged_chunk: ToolCallChunk = {
    "name": (chunk1["name"] or "") + (chunk2["name"] or ""), # Merge tool name
    "args": (chunk1["args"] or "") + (chunk2["args"] or ""), # Merge JSON arguments
    "id": chunk1["id"] or chunk2["id"], # Preserve available ID
    "index": chunk1["index"], # Preserve matching index
    "type": "tool_call_chunk"
}


print("Name:", merged_chunk["name"])
print("Arguments:", merged_chunk["args"])
print("ID:", merged_chunk["id"])
print("Index:", merged_chunk["index"])
print("Type:", merged_chunk["type"])

Name: add_numbers
Arguments: {"first": 10,"second": 20}
ID: call-101
Index: 0
Type: tool_call_chunk



# Functions

1. `tool_call`:= Creates a standardized `ToolCall`.

   ```python
   tool_call(
       *,
       name: str, # Name of the tool to invoke
       args: dict[str, Any], # Arguments passed to the tool
       id: str | None # Identifier associated with the tool call
   ) -> ToolCall
   ```

2. `tool_call_chunk`:= Creates a standardized `ToolCallChunk`.

   ```python
   tool_call_chunk(
       *,
       name: str | None = None, # Partial or complete tool name
       args: str | None = None, # Partial JSON argument string
       id: str | None = None, # Tool-call identifier
       index: int | None = None # Position of the tool call in a sequence
   ) -> ToolCallChunk
   ```

3. `invalid_tool_call`:= Creates an `InvalidToolCall` for a tool call that could not be parsed.

   ```python
   invalid_tool_call(
       *,
       name: str | None = None, # Tool name when available
       args: str | None = None, # Unparsed tool arguments
       id: str | None = None, # Tool-call identifier
       error: str | None = None # Parsing or validation error
   ) -> InvalidToolCall
   ```

4. `default_tool_parser`:= Performs best-effort parsing of raw tool-call dictionaries.

   Successfully decoded JSON arguments produce `ToolCall` objects. Tool calls containing invalid JSON arguments produce `InvalidToolCall` objects. Entries without a `function` field are ignored.

   ```python
   default_tool_parser(
       raw_tool_calls: list[dict[str, Any]] # Raw tool-call dictionaries to parse
   ) -> tuple[list[ToolCall], list[InvalidToolCall]]
   ```

5. `default_tool_chunk_parser`:= Performs best-effort parsing of raw streaming tool-call dictionaries.

   It extracts the function name, argument string, identifier, and index from each raw tool-call chunk.

   ```python
   default_tool_chunk_parser(
       raw_tool_calls: list[dict[str, Any]] # Raw streaming tool-call dictionaries
   ) -> list[ToolCallChunk]
   ```

In [7]:
from pprint import pprint

from langchain_core.messages.tool import (
    default_tool_chunk_parser,
    default_tool_parser,
    invalid_tool_call,
    tool_call,
    tool_call_chunk,
)


# 1. Create a valid ToolCall
valid_call = tool_call(
    name="add_numbers", # Name of the tool
    args={"first": 10, "second": 20}, # Tool arguments
    id="call-101" # Tool-call identifier
)

print("ToolCall:")
pprint(valid_call)


# 2. Create a partial ToolCallChunk
partial_call = tool_call_chunk(
    name="add_numbers", # Partial or complete tool name
    args='{"first": 10,', # Partial JSON arguments
    id="call-102", # Tool-call identifier
    index=0 # Position in the tool-call sequence
)

print("\nToolCallChunk:")
pprint(partial_call)


# 3. Create an InvalidToolCall
invalid_call = invalid_tool_call(
    name="divide_numbers", # Tool name
    args='{"first": 10, "second": }', # Invalid JSON arguments
    id="call-103", # Tool-call identifier
    error="Invalid JSON arguments" # Parsing error
)

print("\nInvalidToolCall:")
pprint(invalid_call)


# 4. Parse raw tool calls
raw_tool_calls = [
    {
        "id": "call-201",
        "function": {
            "name": "multiply_numbers",
            "arguments": '{"first": 4, "second": 5}'
        }
    },
    {
        "id": "call-202",
        "function": {
            "name": "divide_numbers",
            "arguments": '{"first": 10, "second": }'
        }
    }
]

parsed_calls, invalid_calls = default_tool_parser(
    raw_tool_calls # Raw tool-call dictionaries
)

print("\nParsed ToolCalls:")
pprint(parsed_calls)

print("\nInvalid ToolCalls:")
pprint(invalid_calls)


# 5. Parse raw streaming tool-call chunks
raw_tool_call_chunks = [
    {
        "id": "call-301",
        "index": 0,
        "function": {
            "name": "add_numbers",
            "arguments": '{"first": 10,'
        }
    },
    {
        "id": None,
        "index": 0,
        "function": {
            "name": None,
            "arguments": '"second": 20}'
        }
    }
]

parsed_chunks = default_tool_chunk_parser(
    raw_tool_call_chunks # Raw streaming tool-call dictionaries
)

print("\nParsed ToolCallChunks:")
pprint(parsed_chunks)

ToolCall:
{'args': {'first': 10, 'second': 20},
 'id': 'call-101',
 'name': 'add_numbers',
 'type': 'tool_call'}

ToolCallChunk:
{'args': '{"first": 10,',
 'id': 'call-102',
 'index': 0,
 'name': 'add_numbers',
 'type': 'tool_call_chunk'}

InvalidToolCall:
{'args': '{"first": 10, "second": }',
 'error': 'Invalid JSON arguments',
 'id': 'call-103',
 'name': 'divide_numbers',
 'type': 'invalid_tool_call'}

Parsed ToolCalls:
[{'args': {'first': 4, 'second': 5},
  'id': 'call-201',
  'name': 'multiply_numbers',
  'type': 'tool_call'}]

Invalid ToolCalls:
[{'args': '{"first": 10, "second": }',
  'error': None,
  'id': 'call-202',
  'name': 'divide_numbers',
  'type': 'invalid_tool_call'}]

Parsed ToolCallChunks:
[{'args': '{"first": 10,',
  'id': 'call-301',
  'index': 0,
  'name': 'add_numbers',
  'type': 'tool_call_chunk'},
 {'args': '"second": 20}',
  'id': None,
  'index': 0,
  'name': None,
  'type': 'tool_call_chunk'}]
